# **Test d'intégration du loader/Module/InfoNCELoss pour la phase patient-wise-learning**

In [1]:
%load_ext autoreload
%autoreload 2
from gbmhackathon.training.patientwise import *
from gbmhackathon.models.mme import MultiModalEncoder
from gbmhackathon.utils.loss_functions import InfoNCELoss
from gbmhackathon.s3_loader import load_s3
from copy import deepcopy
import numpy as np
import torch
from torch.utils.data import DataLoader
from torch.optim import Adam

In [2]:
name_emb_dict = {"hne":"2025-03-22_12-27_phikon_emb_V1.pkl",
# "spatial":"2025-03-23_18-32_spatial_emb_V1.pkl", #Pour l'instant problm avec le spatial = pas dimensions constantes
"clinical":"2025-03-30_14-23_clinical_emb_V1.pkl",
"wes":"2025-04-05_13-40_wes_emb_V1.pkl"}
pkl_storage_folder = "embedding_V1"

In [3]:
dataset = PatientLearningDataset(name_emb_dict, pkl_storage_folder)
BATCH_SIZE = 20
dataloader = DataLoader(dataset, BATCH_SIZE, shuffle=True, collate_fn=collate_patient_wise)

## Pour la loss, il nous faut un dictionnaire qui associe chaque patient à un ID
On fait bien attention à indiquer les rechutes comme le même patient

In [4]:
missing_mods = load_s3("s3://abstra-project-storage-lttemftb/1b75dc89-ad27-4a65-9e7f-877d1b4f36fc/missing_mod_per_samples.pkl")

In [5]:
all_ids = list(missing_mods.keys())
patient_map = {key: i for key, i in zip(all_ids, [k for k in range(1,len(all_ids)+1)])}
for patient_id, idx in patient_map.items():
    if patient_id.endswith('b'): # pour les rechutes on met le même index que le sample original
        patient_map[patient_id] = idx - 1
patient_map

{'HK_G_001a': 1,
 'HK_G_002a': 2,
 'HK_G_003a': 3,
 'HK_G_004a': 4,
 'HK_G_005a': 5,
 'HK_G_006a': 6,
 'HK_G_007a': 7,
 'HK_G_008a': 8,
 'HK_G_009a': 9,
 'HK_G_010a': 10,
 'HK_G_011a': 11,
 'HK_G_012a': 12,
 'HK_G_013a': 13,
 'HK_G_014a': 14,
 'HK_G_015a': 15,
 'HK_G_016a': 16,
 'HK_G_017b': 16,
 'HK_G_018a': 18,
 'HK_G_019a': 19,
 'HK_G_020a': 20,
 'HK_G_021a': 21,
 'HK_G_022a': 22,
 'HK_G_023a': 23,
 'HK_G_024a': 24,
 'HK_G_025a': 25,
 'HK_G_026a': 26,
 'HK_G_027a': 27,
 'HK_G_028a': 28,
 'HK_G_029b': 28,
 'HK_G_030a': 30,
 'HK_G_031a': 31,
 'HK_G_032a': 32,
 'HK_G_033a': 33,
 'HK_G_034a': 34,
 'HK_G_035a': 35,
 'HK_G_036b': 35,
 'HK_G_037a': 37,
 'HK_G_038a': 38,
 'HK_G_039a': 39,
 'HK_G_040a': 40,
 'HK_G_041a': 41,
 'HK_G_042a': 42,
 'HK_G_043a': 43,
 'HK_G_044b': 43,
 'HK_G_045a': 45,
 'HK_G_046a': 46,
 'HK_G_047a': 47,
 'HK_G_048a': 48,
 'HK_G_049a': 49,
 'HK_G_050a': 50,
 'HK_G_051a': 51,
 'HK_G_052a': 52,
 'HK_G_053a': 53,
 'HK_G_054a': 54,
 'HK_G_055a': 55,
 'HK_G_056a': 56,
 

## Voir à quoi ressemble l'output du loader

In [6]:
toy_batch = next(iter(dataloader))
toy_batch

(['HK_G_084b',
  'HK_G_008a',
  'HK_G_100b',
  'HK_G_036b',
  'HK_G_082b',
  'HK_G_061b',
  'HK_G_017b',
  'HK_G_091a',
  'HK_G_049a',
  'HK_G_104a',
  'HK_G_112a',
  'HK_G_089a',
  'HK_G_011a',
  'HK_G_033a',
  'HK_G_050a',
  'HK_G_016a',
  'HK_G_105b',
  'HK_G_044b',
  'HK_G_042a',
  'HK_G_076a'],
 ['hne', 'clinical', 'wes'],
 {'hne': tensor([[[ 0.0254, -0.0866, -0.1224,  ...,  0.0411,  0.2472,  0.0701]],
  
          [[ 0.1686, -0.4579, -0.2023,  ...,  0.3308,  0.0276, -0.0008]],
  
          [[ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000,  0.0000]],
  
          ...,
  
          [[-0.3093, -0.3896, -0.0976,  ...,  0.0817,  0.1240, -0.1755]],
  
          [[-0.1069, -0.5486,  0.0960,  ...,  0.1254, -0.0653, -0.0451]],
  
          [[ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000,  0.0000]]]),
  'clinical': tensor([[-1.1115e+00,  2.8624e-01, -1.5572e-01,  4.6190e-01, -7.1712e-01,
            2.2223e-01,  4.2495e-02,  1.3009e-01, -9.8888e-02,  4.9125e-02,
           -1.2781e-

In [7]:
raw_emb = toy_batch[2]
for mod in raw_emb.keys():
    print(mod, raw_emb[mod].size())

hne torch.Size([20, 1, 1024])
clinical torch.Size([20, 12])
wes torch.Size([20, 1790])


# Pour instantier le MultiModalEncoder
Il faut des config pour chaque modalité, par simplicité on va définir une coquille de base et simplement ajouter la bonne dimension en entrée

In [8]:
base_config = {"layers": [128,64],
        "dropout": 0.5,
        "act_fn":torch.nn.ReLU,
        "norm_layer":None}

def adapt_base_config(base_cfg, input_size):
    base_copy = deepcopy(base_cfg)
    base_copy["layers"] = [input_size] + base_copy["layers"]
    return base_copy

hne_cfg = adapt_base_config(base_config, 1024)
clinical_cfg = adapt_base_config(base_config, 12)
wes_cfg = adapt_base_config(base_config, 1790)

In [9]:
mme_cfg = {"hne_cfg":hne_cfg, "clinical_cfg":clinical_cfg, "wes_cfg":wes_cfg}

In [11]:
# 1. Define model encoders + loss
mme = MultiModalEncoder(**mme_cfg)
loss_fn = InfoNCELoss(list(name_emb_dict.keys()), patient_map, temperature=0.07, use_all_positives=False)
optimizer = Adam(
    mme.parameters(),
    lr=1e-3,
)

# 2. Training loop
for epoch in range(10):
    epoch_loss = []
    for i, batch in enumerate(dataloader):
        # batch = (X_dict, patient_ids, available_modalities)
        patient_ids, modalities, X_dict, avail_mods = batch
        # print("patient_ids:", patient_ids)
        # print("modalities:", modalities)
        # print("X_dict:", X_dict)
        # print("avail_mods:", avail_mods)
    
        # print(X_dict)
        encoded = mme(X_dict)
        # print(encoded)
    
        # # 2b. Pack into expected batch for loss
        loss_batch = (encoded, patient_ids, avail_mods)
    
        # # 2c. Compute loss & backward
        loss = loss_fn(loss_batch)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        epoch_loss.append(loss.item())
        print(f"Batch {i} loss: {loss.item():.4f}")
    print(f"\nEpoch {i} loss: {np.mean(epoch_loss):.4f}\n".upper())

Batch 0 loss: 4.9507
Batch 1 loss: 4.1047
Batch 2 loss: 7.1172
Batch 3 loss: 1.7266
Batch 4 loss: 5.9113
Batch 5 loss: 3.5584

EPOCH 5 LOSS: 4.5615

Batch 0 loss: 4.7882
Batch 1 loss: 2.3230
Batch 2 loss: 5.4448
Batch 3 loss: 3.2481
Batch 4 loss: 7.6857
Batch 5 loss: nan

EPOCH 5 LOSS: NAN

Batch 0 loss: nan
Batch 1 loss: nan
Batch 2 loss: nan
Batch 3 loss: nan
Batch 4 loss: nan
Batch 5 loss: nan

EPOCH 5 LOSS: NAN

Batch 0 loss: nan
Batch 1 loss: nan
Batch 2 loss: nan
Batch 3 loss: nan
Batch 4 loss: nan
Batch 5 loss: nan

EPOCH 5 LOSS: NAN

Batch 0 loss: nan
Batch 1 loss: nan
Batch 2 loss: nan
Batch 3 loss: nan
Batch 4 loss: nan
Batch 5 loss: nan

EPOCH 5 LOSS: NAN

Batch 0 loss: nan
Batch 1 loss: nan
Batch 2 loss: nan
Batch 3 loss: nan
Batch 4 loss: nan
Batch 5 loss: nan

EPOCH 5 LOSS: NAN

Batch 0 loss: nan
Batch 1 loss: nan
Batch 2 loss: nan
Batch 3 loss: nan
Batch 4 loss: nan
Batch 5 loss: nan

EPOCH 5 LOSS: NAN

Batch 0 loss: nan
Batch 1 loss: nan
Batch 2 loss: nan
Batch 3 loss: 